# 유도탄 오토파일럿 설계 (Missile Autopilot Design)

**유도법칙이 생성한 가속도 명령을 실제로 추종하는 조종 시스템 설계**

유도법칙(Guidance Law)은 매 순간 "얼마나 꺾어야 한다"는 가속도 명령 $a_{cmd}$를 생성합니다.  
오토파일럿(Autopilot, 자동조종장치)은 그 명령을 실제 핀 편향각(fin deflection) $\delta$로 변환하여 유도탄 기체가 명령을 충실히 추종하도록 합니다.

---

## 학습 목표

1. 유도탄 단주기(short-period) 공력 전달함수 이해
2. **2-loop autopilot** 설계 및 안정성 분석 (Blakelock Ch.7)
3. **3-loop autopilot** 설계 및 synthetic stability augmentation (Garnell Ch.6)
4. 핀 구동기(Fin Actuator) 모델 및 대역폭 제약
5. **Gain Scheduling** — 비행 조건 변화에 따른 이득 적응
6. 비선형 시간영역 시뮬레이션

---

> **참고 문헌**  
> - Blakelock, J.H., *Automatic Control of Aircraft and Missiles*, Wiley, 1991  
> - Garnell, P., *Guided Weapon Control Systems*, 2nd Ed., Brassey's, 1980

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal
from scipy.signal import TransferFunction, bode, step

# 한글 폰트 설정
plt.rcParams['font.family'] = ['AppleGothic', 'Malgun Gothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("패키지 로드 완료.")

---
## 1. 유도탄 공력 전달함수 (Airframe Aerodynamic Transfer Function)

### 단주기 근사 (Short-Period Approximation)

유도탄의 종방향 운동은 받음각 $\alpha$와 피치율 $q$를 상태변수로 하는 2차 연립미분방정식으로 표현됩니다.

$$\dot{\alpha} = Z_\alpha \alpha + q + Z_\delta \delta$$
$$\dot{q} = M_\alpha \alpha + M_q q + M_\delta \delta$$

법선 가속도 (normal acceleration):
$$a_z = V(Z_\alpha \alpha + Z_\delta \delta)$$

### 주요 안정성 미분계수 (Stability Derivatives)

| 기호 | 물리적 의미 | 단위 |
|------|------------|------|
| $M_\alpha$ | 받음각에 의한 피칭 모멘트 (정적 안정성) | 1/s² |
| $M_q$ | 피치율에 의한 피칭 모멘트 (피치 감쇠) | 1/s |
| $M_\delta$ | 핀 편향에 의한 피칭 모멘트 (조종력) | 1/s² |
| $Z_\alpha$ | 받음각에 의한 법선력 | 1/s |
| $Z_\delta$ | 핀 편향에 의한 법선력 | 1/s |

### 핀 입력 → 법선 가속도 전달함수

라플라스 변환을 적용하면:

$$\frac{a_z(s)}{\delta(s)} = \frac{V[Z_\delta(s - M_q) + M_\delta Z_\alpha - M_\alpha Z_\delta]}{s^2 - (Z_\alpha + M_q)s + (Z_\alpha M_q - M_\alpha)}$$

분모의 판별식: $\Delta = (Z_\alpha + M_q)^2 - 4(Z_\alpha M_q - M_\alpha)$

---

### 안정 vs 불안정 미사일

- **안정한 미사일** ($M_\alpha < 0$): 압력중심(CP)이 무게중심(CG) 후방 → 자기복원력 존재  
  → 오토파일럿 설계 상대적으로 용이
- **불안정한 미사일** ($M_\alpha > 0$): CP가 CG 전방 → 자기발산 경향  
  → 능동 안정화 필수 (active stabilization), 3-loop 구조가 유리

In [ ]:
# ---------------------------------------------------------------
# 1. 공력 전달함수 파라미터 정의 (Mach 2.0, 해수면 기준)
# autopilot.py의 AirframeShortPeriod 클래스 참조
# ---------------------------------------------------------------

# 공력 미분계수 (안정한 미사일, codebase convention: 음수 = 안정)
M_alpha = -200.0   # /s^2  정적 안정성 (static stability)
M_q     =  -4.0   # /s    피치 감쇠 (pitch damping)
M_delta = -80.0   # /s^2  조종 효율 (control effectiveness)
Z_alpha =  -3.0   # /s    법선력 기울기 (normal force slope)
Z_delta =  -0.2   # /s    핀에 의한 법선력 (fin normal force)
V       = 680.0   # m/s   비행속도 (Mach 2.0 @ SL)

print("=== 공력 미분계수 (Mach 2.0) ===")
print(f"  M_α = {M_alpha:8.1f} /s²  (정적 안정성: {'안정' if M_alpha < 0 else '불안정'})")
print(f"  M_q = {M_q:8.1f} /s   (피치 감쇠)")
print(f"  M_δ = {M_delta:8.1f} /s²  (조종 효율)")
print(f"  Z_α = {Z_alpha:8.1f} /s   (법선력 기울기)")
print(f"  Z_δ = {Z_delta:8.1f} /s   (핀 법선력)")
print(f"  V   = {V:8.1f} m/s  (비행속도)")

# 전달함수 수치 유도: a_z(s) / delta(s)
# 분자: V * [Z_delta*(s - M_q) + M_delta*Z_alpha - M_alpha*Z_delta]
#      = V * [Z_delta*s - Z_delta*M_q + M_delta*Z_alpha - M_alpha*Z_delta]
b1 = V * Z_delta
b0 = V * (-Z_delta * M_q + M_delta * Z_alpha - M_alpha * Z_delta)

# 분모: s^2 - (Z_alpha + M_q)*s + (Z_alpha*M_q - M_alpha)
a2 = 1.0
a1 = -(Z_alpha + M_q)
a0 = Z_alpha * M_q - M_alpha

# 단주기 고유진동수 및 감쇠비
omega_sp = np.sqrt(abs(a0))           # 단주기 고유진동수 (rad/s)
zeta_sp  = a1 / (2.0 * omega_sp)     # 단주기 감쇠비

print(f"\n=== 단주기 동특성 ===")
print(f"  ωsp = {omega_sp:.2f} rad/s  (단주기 고유진동수)")
print(f"  ζsp = {zeta_sp:.3f}          (단주기 감쇠비)")
print(f"  분자 계수: b1={b1:.2f}, b0={b0:.2f}")
print(f"  분모 계수: a2={a2:.2f}, a1={a1:.2f}, a0={a0:.2f}")

# scipy.signal 전달함수 생성
TF_airframe = TransferFunction([b1, b0], [a2, a1, a0])
print(f"\n전달함수 az/delta:")
print(f"  분자: {TF_airframe.num}")
print(f"  분모: {TF_airframe.den}")

In [ ]:
# ---------------------------------------------------------------
# 기체 전달함수 보드 선도 (Open-loop Airframe Bode Plot)
# ---------------------------------------------------------------
w_bode = np.logspace(-1, 3, 2000)   # 0.1 ~ 1000 rad/s
w_af, mag_af, phase_af = bode(TF_airframe, w=w_bode)

fig, axes = plt.subplots(2, 1, figsize=(10, 7))
fig.suptitle('기체 전달함수 보드 선도  $a_z(s)/\\delta(s)$  (Mach 2.0, 해수면)', fontsize=13)

axes[0].semilogx(w_af, mag_af, 'b-', linewidth=2)
axes[0].set_ylabel('크기 (dB)', fontsize=11)
axes[0].set_title('크기 선도 (Magnitude)', fontsize=10)
axes[0].axvline(omega_sp, color='r', linestyle='--', alpha=0.7, label=f'ωsp = {omega_sp:.1f} rad/s')
axes[0].legend(fontsize=9)

axes[1].semilogx(w_af, phase_af, 'g-', linewidth=2)
axes[1].set_ylabel('위상 (deg)', fontsize=11)
axes[1].set_xlabel('주파수 (rad/s)', fontsize=11)
axes[1].set_title('위상 선도 (Phase)', fontsize=10)
axes[1].axhline(-180, color='k', linestyle=':', alpha=0.5)
axes[1].axvline(omega_sp, color='r', linestyle='--', alpha=0.7, label=f'ωsp = {omega_sp:.1f} rad/s')
axes[1].legend(fontsize=9)

for ax in axes:
    ax.set_xlim([0.1, 1000])
    ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. 2-Loop Autopilot

### 구조 (Blakelock Ch.7 Classic 2-Loop)

```
a_cmd ──(+)──[Kp + Ki/s]──(+)──[Actuator]──[Airframe]──── a_z
         │(-) 외부루프         │(-)                           │
         └─────────────────── a_measured ──────────────────┘
                              내부루프: rate gyro            │
                              └──────── Kq * q_meas ────────┘
```

- **내부 루프 (Inner Loop)**: 레이트 자이로(rate gyro)로 피치율 $q$를 측정 → 이득 $K_q$ 피드백으로 감쇠 증대 (damping augmentation)  
  유효 피치 감쇠: $M_q^{eff} = M_q + M_\delta K_q$

- **외부 루프 (Outer Loop)**: PI 제어기로 가속도 오차 추종 (proportional-integral)

핀 명령:
$$\delta_{cmd} = \underbrace{K_p \cdot e_a + K_i \int e_a \, dt}_{\text{외부 PI}} - K_q \cdot q$$

여기서 $e_a = a_{cmd} - a_{measured}$

### 이득 선정 원칙
- $K_q$: 내부 루프 감쇠비 $\zeta_{inner} \approx 0.7$ 목표로 선정  
- $K_p$, $K_i$: 외부 루프 대역폭 및 정상상태 오차 = 0 달성  
- 외부/내부 루프 대역폭 분리: $\omega_{outer} \ll \omega_{inner}$ (통상 3~5배 분리)

In [ ]:
# ---------------------------------------------------------------
# 2-Loop Autopilot 전달함수 해석
# autopilot.py의 TwoLoopAutopilot 파라미터 기반
# ---------------------------------------------------------------

# 검증된 이득 (Mach 2.0, 해수면)
Kq = -0.20   # 레이트 피드백 이득 (M_delta < 0 규약: 음수로 감쇠 증대)
Kp =  0.001  # 비례 이득 (proportional gain)
Ki =  0.01   # 적분 이득 (integral gain)

# --- 내부 루프 닫힌 전달함수 ---
# delta = -Kq*q + outer_cmd
# 내부 루프 (rate feedback loop)의 등가 전달함수
# M_q_eff = M_q - M_delta * Kq  (Kq < 0 이므로 더 음수 → 더 많은 감쇠)
M_q_eff = M_q - M_delta * Kq
print(f"내부 루프 적용 후:")
print(f"  M_q (원래)     = {M_q:.2f} /s")
print(f"  M_q_eff (증대) = {M_q_eff:.2f} /s  (감쇠 증대 확인)")

# 내부 루프 닫힌 후 기체 전달함수 (q 루프 포함)
# 분모: s^2 - (Z_alpha + M_q_eff)*s + (Z_alpha*M_q_eff - M_alpha)
a1_inner = -(Z_alpha + M_q_eff)
a0_inner = Z_alpha * M_q_eff - M_alpha
omega_inner = np.sqrt(abs(a0_inner))
zeta_inner  = a1_inner / (2.0 * omega_inner)
print(f"\n내부 루프 닫힌 후 단주기 동특성:")
print(f"  ωn = {omega_inner:.2f} rad/s")
print(f"  ζ  = {zeta_inner:.3f} (목표: ~0.7)")

# 내부 루프 닫힌 후 기체 전달함수
# 분자는 동일: b1*s + b0 (rate gyro는 q만 측정, az 분자에 영향 없음)
# 단, 분모 계수가 변경됨
TF_inner_closed = TransferFunction([b1, b0], [a2, a1_inner, a0_inner])

# --- 외부 루프: PI 제어기 ---
# C(s) = Kp + Ki/s = (Kp*s + Ki) / s
TF_PI = TransferFunction([Kp, Ki], [1.0, 0.0])

# 외부 루프 개루프 전달함수: C(s) * G_inner(s)
# scipy의 series() 사용
TF_OL_outer = signal.series(TF_PI.num, TF_PI.den,
                             TF_inner_closed.num, TF_inner_closed.den)

# 외부 루프 닫힌 전달함수 (unity feedback)
TF_CL_2loop = signal.feedback(TF_OL_outer[0], TF_OL_outer[1], sign=-1)

print(f"\n2-Loop 닫힌 전달함수:")
print(f"  분자: {np.round(TF_CL_2loop[0], 4)}")
print(f"  분모: {np.round(TF_CL_2loop[1], 4)}")

# --- 계단 응답 ---
t_step = np.linspace(0, 1.0, 5000)
a_cmd_val = 40.0   # 40 m/s^2 (≈ 4g) 명령

t_out, y_out = signal.step(TransferFunction(TF_CL_2loop[0], TF_CL_2loop[1]), T=t_step)
y_out_scaled = y_out * a_cmd_val

# 상승시간, 정착시간 계산
y_final = y_out_scaled[-1]
idx_rise = np.where(y_out_scaled >= 0.9 * y_final)[0]
t_rise = t_out[idx_rise[0]] if len(idx_rise) > 0 else np.nan
idx_settle = np.where(np.abs(y_out_scaled - y_final) > 0.02 * abs(y_final))[0]
t_settle = t_out[idx_settle[-1]] if len(idx_settle) > 0 else 0.0

print(f"\n=== 2-Loop 계단 응답 성능 (명령: {a_cmd_val} m/s²) ===")
print(f"  상승 시간 (90%)  : {t_rise*1000:.1f} ms")
print(f"  정착 시간 (±2%)  : {t_settle*1000:.1f} ms")
print(f"  정상상태 추종값  : {y_final:.2f} m/s² (오차: {abs(y_final - a_cmd_val)/a_cmd_val*100:.2f}%)")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_out * 1000, y_out_scaled, 'b-', linewidth=2, label='달성 가속도')
ax.axhline(a_cmd_val, color='r', linestyle='--', linewidth=1.5, label=f'명령 ({a_cmd_val} m/s²)')
ax.axhline(0.9 * a_cmd_val, color='gray', linestyle=':', alpha=0.6, label='90% 선')
ax.set_xlabel('시간 (ms)', fontsize=11)
ax.set_ylabel('법선 가속도 (m/s²)', fontsize=11)
ax.set_title('2-Loop Autopilot 계단 응답 (Step Response)', fontsize=12)
ax.legend(fontsize=10)
ax.set_xlim([0, 500])
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# 2-Loop 보드 선도 — 이득 여유 / 위상 여유 분석
# ---------------------------------------------------------------
w_bode2 = np.logspace(-1, 3, 3000)

# 개루프 전달함수 보드
TF_OL_sys = TransferFunction(TF_OL_outer[0], TF_OL_outer[1])
w_ol, mag_ol, phase_ol = bode(TF_OL_sys, w=w_bode2)

# 닫힌 루프 전달함수 보드
TF_CL_sys = TransferFunction(TF_CL_2loop[0], TF_CL_2loop[1])
w_cl, mag_cl, phase_cl = bode(TF_CL_sys, w=w_bode2)

# 이득 교차 주파수 (Gain Crossover Frequency): |G(jw)| = 0 dB
gc_idx = np.where(np.diff(np.sign(mag_ol)))[0]
if len(gc_idx) > 0:
    w_gc = w_ol[gc_idx[0]]
    pm = phase_ol[gc_idx[0]] + 180.0
    print(f"이득 교차 주파수: {w_gc:.2f} rad/s")
    print(f"위상 여유 (Phase Margin): {pm:.1f} deg  ({'양호' if pm > 30 else '주의 필요'})")
else:
    w_gc, pm = None, None
    print("이득 교차 주파수: 해당 없음")

# 위상 교차 주파수 (Phase Crossover Frequency): ∠G(jw) = -180 deg
pc_idx = np.where(np.diff(np.sign(phase_ol + 180.0)))[0]
if len(pc_idx) > 0:
    w_pc = w_ol[pc_idx[0]]
    gm = -mag_ol[pc_idx[0]]
    print(f"위상 교차 주파수: {w_pc:.2f} rad/s")
    print(f"이득 여유 (Gain Margin):  {gm:.1f} dB  ({'양호' if gm > 6 else '주의 필요'})")
else:
    w_pc, gm = None, None
    print("위상 교차 주파수: 해당 없음")

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('2-Loop Autopilot 보드 선도 분석', fontsize=13)

# 개루프 크기
axes[0, 0].semilogx(w_ol, mag_ol, 'b-', linewidth=2)
axes[0, 0].axhline(0, color='k', linestyle='--', alpha=0.5)
if w_gc is not None:
    axes[0, 0].axvline(w_gc, color='r', linestyle=':', alpha=0.7, label=f'ωgc={w_gc:.1f}')
    axes[0, 0].legend(fontsize=9)
axes[0, 0].set_ylabel('크기 (dB)')
axes[0, 0].set_title('개루프 크기 (Open-Loop Magnitude)')

# 개루프 위상
axes[1, 0].semilogx(w_ol, phase_ol, 'g-', linewidth=2)
axes[1, 0].axhline(-180, color='k', linestyle='--', alpha=0.5, label='-180°')
if w_gc is not None:
    axes[1, 0].axvline(w_gc, color='r', linestyle=':', alpha=0.7)
    axes[1, 0].annotate(f'PM={pm:.0f}°', xy=(w_gc, phase_ol[gc_idx[0]]),
                        xytext=(w_gc * 2, phase_ol[gc_idx[0]] - 20),
                        fontsize=9, color='red',
                        arrowprops=dict(arrowstyle='->', color='red'))
axes[1, 0].legend(fontsize=9)
axes[1, 0].set_xlabel('주파수 (rad/s)')
axes[1, 0].set_ylabel('위상 (deg)')
axes[1, 0].set_title('개루프 위상 (Open-Loop Phase)')

# 닫힌 루프 크기
axes[0, 1].semilogx(w_cl, mag_cl, 'b-', linewidth=2)
axes[0, 1].axhline(-3, color='orange', linestyle='--', alpha=0.7, label='-3 dB')
bw_idx = np.where(mag_cl <= -3.0)[0]
if len(bw_idx) > 0:
    w_bw = w_cl[bw_idx[0]]
    axes[0, 1].axvline(w_bw, color='orange', linestyle=':', alpha=0.7,
                       label=f'BW={w_bw:.1f} rad/s')
axes[0, 1].legend(fontsize=9)
axes[0, 1].set_ylabel('크기 (dB)')
axes[0, 1].set_title('닫힌 루프 크기 (Closed-Loop Magnitude)')

# 닫힌 루프 위상
axes[1, 1].semilogx(w_cl, phase_cl, 'g-', linewidth=2)
axes[1, 1].set_xlabel('주파수 (rad/s)')
axes[1, 1].set_ylabel('위상 (deg)')
axes[1, 1].set_title('닫힌 루프 위상 (Closed-Loop Phase)')

for ax in axes.flat:
    ax.set_xlim([0.1, 500])
    ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. 3-Loop Autopilot

### 2-Loop 대비 Synthetic Stability Augmentation 추가

3-loop 구조는 Garnell Ch.6 및 Abd-Elatif et al.에서 제안된 구조로,  
**정적 불안정 미사일 ($M_\alpha > 0$)** 에서도 안정화가 가능합니다.

```
a_cmd ─(+)─[KA + Ki/s]─(+)─[Actuator]─[Airframe]── a_z
        │(-) 외부루프    │(+)                          │
        └── a_meas ─────┤                              │
                    (K_omega + Kg) * q_meas ───────────┘
                    ↑           ↑
                  중간루프    내부루프
                (synthetic  (rate gyro
                 stability)  damping)
```

### 3개 중첩 루프 구조

| 루프 | 센서 | 이득 | 역할 |
|------|------|------|------|
| 내부 (Inner) | 레이트 자이로 $q$ | $K_g$ | 피치 감쇠 증대 |
| 중간 (Middle) | 레이트 자이로 $q$ | $K_\omega$ | Synthetic stability (안정화) |
| 외부 (Outer) | 가속도계 $a_z$ | $K_A + K_i/s$ | 가속도 명령 추종 |

### 이득 설계 공식 (three_loop_autopilot.py 기반)

$$K_g = \frac{-2\zeta\omega_n - M_q}{M_\delta}$$

$$K_\omega = 0.1 \cdot |K_g|$$

$$K_A = \frac{1}{\tau \cdot |Z_\alpha| \cdot V}$$

- $\omega_n$: 목표 폐루프 대역폭 (rad/s)
- $\zeta$: 목표 감쇠비 (통상 0.7)
- $\tau$: 목표 시정수 (s)

> **Garnell의 핵심 통찰**: 중간 루프의 $K_\omega$는 CP-CG 분리로 인한 정적 불안정성을 능동적으로 보상합니다.  
> 이를 *synthetic stability augmentation* 이라 부릅니다.

In [ ]:
# ---------------------------------------------------------------
# 3-Loop Autopilot 이득 계산
# three_loop_autopilot.py의 ThreeLoopAutopilot 참조
# ---------------------------------------------------------------

# 3-loop 설계 파라미터
omega_3L = 20.0   # 목표 대역폭 (rad/s)
zeta_3L  = 0.7    # 목표 감쇠비
tau_3L   = 0.5    # 목표 시정수 (s)
Ki_3L    = 0.02   # 적분 이득
KDC      = 1.0    # DC 이득 (정상상태 추종)

# 이득 계산
Kg      = (-2.0 * zeta_3L * omega_3L - M_q) / M_delta
K_omega = 0.1 * abs(Kg)
KA      = 1.0 / (tau_3L * abs(Z_alpha) * V)

print("=== 3-Loop Autopilot 이득 ===")
print(f"  Kg      = {Kg:.5f} rad / (rad/s)  (내부 루프: 레이트 감쇠)")
print(f"  K_omega = {K_omega:.5f} rad / (rad/s)  (중간 루프: synthetic stability)")
print(f"  KA      = {KA:.6f} rad / (m/s²)   (외부 루프: 가속도 피드백)")
print(f"  Ki      = {Ki_3L:.4f}                   (적분 이득)")
print(f"  KDC     = {KDC:.2f}                     (DC 이득)")

# 유효 피치 감쇠 (내부 + 중간 루프 합산)
K_total = K_omega + Kg
M_q_eff_3L = M_q + M_delta * K_total   # 3-loop 등가 M_q
print(f"\n  Kg + K_omega = {K_total:.5f}")
print(f"  M_q_eff (3-loop) = {M_q_eff_3L:.2f} /s  (2-loop: {M_q_eff:.2f})")

# 내부+중간 루프 닫힌 후 분모 계수
a1_3L = -(Z_alpha + M_q_eff_3L)
a0_3L = Z_alpha * M_q_eff_3L - M_alpha

# 내부+중간 루프 닫힌 후 기체 전달함수
TF_inner_3L = TransferFunction([b1, b0], [a2, a1_3L, a0_3L])

# 외부 루프: KA (비례) + Ki/s (적분)
TF_outer_3L = TransferFunction([KA * KDC, Ki_3L], [1.0, 0.0])

# 개루프
TF_OL_3L = signal.series(TF_outer_3L.num, TF_outer_3L.den,
                          TF_inner_3L.num, TF_inner_3L.den)

# 닫힌 루프
TF_CL_3loop = signal.feedback(TF_OL_3L[0], TF_OL_3L[1], sign=-1)

# 계단 응답
t_3L, y_3L = signal.step(TransferFunction(TF_CL_3loop[0], TF_CL_3loop[1]), T=t_step)
y_3L_scaled = y_3L * a_cmd_val

y_final_3L = y_3L_scaled[-1]
idx_rise_3L = np.where(y_3L_scaled >= 0.9 * abs(y_final_3L))[0]
t_rise_3L = t_3L[idx_rise_3L[0]] * 1000 if len(idx_rise_3L) > 0 else np.nan

print(f"\n3-Loop 정상상태 추종값: {y_final_3L:.2f} m/s²")
print(f"3-Loop 상승 시간 (90%): {t_rise_3L:.1f} ms")

# 2-loop vs 3-loop 계단 응답 비교
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_out * 1000, y_out_scaled, 'b-', linewidth=2, label='2-Loop')
ax.plot(t_3L * 1000, y_3L_scaled, 'r-', linewidth=2, label='3-Loop')
ax.axhline(a_cmd_val, color='k', linestyle='--', linewidth=1.5, label=f'명령 ({a_cmd_val} m/s²)')
ax.axhline(0.9 * a_cmd_val, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('시간 (ms)', fontsize=11)
ax.set_ylabel('법선 가속도 (m/s²)', fontsize=11)
ax.set_title('2-Loop vs 3-Loop Autopilot 계단 응답 비교', fontsize=12)
ax.legend(fontsize=11)
ax.set_xlim([0, 600])
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# 2-Loop vs 3-Loop 보드 선도 비교
# ---------------------------------------------------------------
w_cmp = np.logspace(-1, 3, 3000)

# 닫힌 루프 보드 선도
_, mag_2L_cl, phase_2L_cl = bode(TransferFunction(TF_CL_2loop[0], TF_CL_2loop[1]), w=w_cmp)
_, mag_3L_cl, phase_3L_cl = bode(TransferFunction(TF_CL_3loop[0], TF_CL_3loop[1]), w=w_cmp)

# -3dB 대역폭
def find_bandwidth(w, mag_db, threshold=-3.0):
    idx = np.where(mag_db <= threshold)[0]
    return w[idx[0]] if len(idx) > 0 else None

bw_2L = find_bandwidth(w_cmp, mag_2L_cl)
bw_3L = find_bandwidth(w_cmp, mag_3L_cl)

print(f"=== 대역폭 비교 ===")
print(f"  2-Loop 대역폭: {bw_2L:.2f} rad/s" if bw_2L else "  2-Loop 대역폭: 측정 불가")
print(f"  3-Loop 대역폭: {bw_3L:.2f} rad/s" if bw_3L else "  3-Loop 대역폭: 측정 불가")

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
fig.suptitle('2-Loop vs 3-Loop 보드 선도 비교 (Closed-Loop)', fontsize=13)

axes[0].semilogx(w_cmp, mag_2L_cl, 'b-', linewidth=2, label='2-Loop')
axes[0].semilogx(w_cmp, mag_3L_cl, 'r-', linewidth=2, label='3-Loop')
axes[0].axhline(-3, color='gray', linestyle='--', alpha=0.7, label='-3 dB')
if bw_2L:
    axes[0].axvline(bw_2L, color='b', linestyle=':', alpha=0.6, label=f'BW_2L={bw_2L:.1f}')
if bw_3L:
    axes[0].axvline(bw_3L, color='r', linestyle=':', alpha=0.6, label=f'BW_3L={bw_3L:.1f}')
axes[0].set_ylabel('크기 (dB)', fontsize=11)
axes[0].set_title('닫힌 루프 크기', fontsize=10)
axes[0].legend(fontsize=9)
axes[0].set_xlim([0.1, 500])
axes[0].set_ylim([-40, 10])

axes[1].semilogx(w_cmp, phase_2L_cl, 'b-', linewidth=2, label='2-Loop')
axes[1].semilogx(w_cmp, phase_3L_cl, 'r-', linewidth=2, label='3-Loop')
axes[1].set_xlabel('주파수 (rad/s)', fontsize=11)
axes[1].set_ylabel('위상 (deg)', fontsize=11)
axes[1].set_title('닫힌 루프 위상', fontsize=10)
axes[1].legend(fontsize=9)
axes[1].set_xlim([0.1, 500])

for ax in axes:
    ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. 핀 구동기 모델 (Fin Actuator)

### 2차 전달함수 모델

핀 구동기는 전기-유압 혹은 전동 액추에이터로, 2차 전달함수로 근사됩니다:

$$\frac{\delta(s)}{\delta_{cmd}(s)} = \frac{\omega_n^2}{s^2 + 2\zeta\omega_n s + \omega_n^2}$$

### 물리적 한계

- **위치 한계 (Position Limit)**: $|\delta| \leq \delta_{max}$ — 핀이 물리적으로 꺾을 수 있는 최대 각도  
- **속도 한계 (Rate Limit)**: $|\dot{\delta}| \leq \dot{\delta}_{max}$ — 구동기 유압/모터 최대 속도

### 설계 규칙: 대역폭 분리 (Bandwidth Separation)

$$\omega_{actuator} \gg \omega_{autopilot} \gg \omega_{guidance}$$

통상: $\omega_{act} \geq 3 \times \omega_{autopilot}$

> **구동기 대역폭이 오토파일럿 성능의 상한 (upper bound)** 을 결정합니다.  
> 구동기가 느리면 아무리 좋은 오토파일럿 알고리즘도 성능이 제한됩니다 (Garnell).

| 파라미터 | 값 | 설명 |
|----------|----|----- |
| $\omega_n$ | 120 rad/s | 자연 주파수 |
| $\zeta$ | 0.7 | 감쇠비 |
| $\delta_{max}$ | 25° | 최대 핀 각도 |
| $\dot{\delta}_{max}$ | 300°/s | 최대 핀 속도 |

In [ ]:
# ---------------------------------------------------------------
# 핀 구동기 모델 및 계단 응답
# actuator.py의 FinActuator 참조
# ---------------------------------------------------------------

wn_act   = 120.0                  # 자연 주파수 (rad/s)
zeta_act = 0.7                    # 감쇠비
delta_max = np.radians(25.0)      # 위치 한계 (rad)
rate_max  = np.radians(300.0)     # 속도 한계 (rad/s)

# 선형 2차 전달함수
TF_act = TransferFunction([wn_act**2], [1.0, 2*zeta_act*wn_act, wn_act**2])

# 계단 응답 (선형 모델)
delta_cmd_val = np.radians(10.0)  # 10도 명령
t_act = np.linspace(0, 0.2, 2000)
t_act_out, y_act_lin = signal.step(TF_act, T=t_act)
y_act_lin_deg = np.degrees(y_act_lin * delta_cmd_val)

# 비선형 시뮬레이션 (속도 한계 포함)
def simulate_actuator_nonlinear(t_arr, delta_cmd_arr, wn, zeta, d_max, r_max):
    """2차 구동기 비선형 시뮬레이션 (RK4, 속도/위치 포화 포함)"""
    d = 0.0
    dd = 0.0
    output = np.zeros_like(t_arr)
    for i, t in enumerate(t_arr):
        dt = t_arr[1] - t_arr[0] if i == 0 else t_arr[i] - t_arr[i-1]
        cmd = delta_cmd_arr[i]
        # RK4
        def deriv(d_, dd_):
            d_dot = dd_
            dd_dot = wn**2 * (cmd - d_) - 2*zeta*wn*dd_
            return d_dot, dd_dot
        k1d, k1dd = deriv(d, dd)
        k2d, k2dd = deriv(d + 0.5*dt*k1d, np.clip(dd + 0.5*dt*k1dd, -r_max, r_max))
        k3d, k3dd = deriv(d + 0.5*dt*k2d, np.clip(dd + 0.5*dt*k2dd, -r_max, r_max))
        k4d, k4dd = deriv(d + dt*k3d,     np.clip(dd + dt*k3dd,     -r_max, r_max))
        d  = np.clip(d + (dt/6)*(k1d + 2*k2d + 2*k3d + k4d), -d_max, d_max)
        dd = np.clip(dd + (dt/6)*(k1dd + 2*k2dd + 2*k3dd + k4dd), -r_max, r_max)
        output[i] = d
    return output

# 소형 명령 (선형 영역, 속도 포화 없음)
cmd_small  = np.full_like(t_act, np.radians(10.0))
# 대형 명령 (속도 포화 발생)
cmd_large  = np.full_like(t_act, np.radians(25.0))

y_nl_small = simulate_actuator_nonlinear(t_act, cmd_small,  wn_act, zeta_act, delta_max, rate_max)
y_nl_large = simulate_actuator_nonlinear(t_act, cmd_large,  wn_act, zeta_act, delta_max, rate_max)

print(f"구동기 모델 파라미터:")
print(f"  ωn = {wn_act} rad/s  (대역폭 기준)")
print(f"  ζ  = {zeta_act}")
print(f"  위치 한계: ±{np.degrees(delta_max):.0f}°")
print(f"  속도 한계: ±{np.degrees(rate_max):.0f}°/s")
print(f"  최대 속도 도달 시간: {np.degrees(delta_max)/np.degrees(rate_max)*1000:.1f} ms (포화 기준)")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('핀 구동기 모델 응답 (Fin Actuator Response)', fontsize=13)

# 선형 vs 비선형 (10도 명령)
axes[0].plot(t_act_out * 1000, y_act_lin_deg, 'b--', linewidth=2, label='선형 모델')
axes[0].plot(t_act * 1000, np.degrees(y_nl_small), 'r-', linewidth=2, label='비선형 (속도 제한 포함)')
axes[0].axhline(10.0, color='k', linestyle=':', alpha=0.6, label='명령 (10°)')
axes[0].set_xlabel('시간 (ms)')
axes[0].set_ylabel('핀 편향각 (deg)')
axes[0].set_title('소형 명령 (10°) — 선형/비선형 비교')
axes[0].legend(fontsize=9)
axes[0].set_xlim([0, 150])

# 대형 명령: 속도 포화 효과
axes[1].plot(t_act * 1000, np.degrees(y_nl_large), 'r-', linewidth=2, label='실제 응답 (속도 포화)')
axes[1].axhline(25.0, color='k', linestyle=':', alpha=0.6, label='명령/위치 한계 (25°)')
axes[1].axhline(np.degrees(rate_max) * (t_act[1] - t_act[0]) * 100,
               color='orange', linestyle='--', alpha=0.6)
# 속도 한계선 (기울기)
t_rate = np.linspace(0, np.degrees(delta_max) / np.degrees(rate_max), 100)
axes[1].plot(t_rate * 1000, np.degrees(rate_max) * t_rate,
            'orange', linestyle='--', linewidth=1.5, label=f'속도 한계 기울기 ({np.degrees(rate_max):.0f}°/s)')
axes[1].set_xlabel('시간 (ms)')
axes[1].set_ylabel('핀 편향각 (deg)')
axes[1].set_title('대형 명령 (25°) — 속도 포화 효과')
axes[1].legend(fontsize=9)
axes[1].set_xlim([0, 200])

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Gain Scheduling (이득 스케줄링)

### 왜 이득 스케줄링이 필요한가?

유도탄은 비행 중 마하수(Mach)와 동압(dynamic pressure)이 급격히 변합니다.  
공력 계수는 이 두 변수에 강하게 의존하므로, 동일한 이득으로는 전 비행 포락선(flight envelope)에서 일정한 성능을 보장할 수 없습니다.

$$q_{bar} = \frac{1}{2} \rho V^2 \propto \text{Mach}^2 \cdot \rho(h)$$

$$M_\alpha = \frac{q_{bar} \cdot S_{ref} \cdot d_{ref} \cdot C_{m\alpha}}{I_{yy}} \propto q_{bar}$$

동압이 증가할수록 공력 미분계수의 크기가 커지고 → 고정 이득 사용 시 시스템이 불안정해질 수 있습니다.

### 스케줄링 변수

- 1차 변수: **마하수 (Mach)**
- 2차 변수: **고도 (Altitude)** → 대기 밀도 $\rho$를 통해 동압 결정

### 구현 방법 (gain_scheduler.py 기반)

1. (Mach, 고도) 격자점마다 공력 계수 테이블 조회
2. 무차원 계수 → 유차원 안정성 미분계수 변환
3. ThreeLoopAutopilot 이득 공식으로 이득 계산
4. 런타임에 쌍선형 보간(bilinear interpolation) 수행

In [ ]:
# ---------------------------------------------------------------
# 이득 스케줄링: 마하수에 따른 이득 변화 시각화
# ---------------------------------------------------------------

# 마하수별 공력 미분계수 모의 (단순 스케일링 모델)
# 실제: gain_scheduler.py의 _nondim_to_dim() 사용
# 여기서는 교육용 단순 모델 사용

mach_range = np.array([1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0])
gamma_air  = 1.4
a_sound_SL = 340.3  # m/s at SL

# 동압 비례 인자 (SL 기준, Mach 2.0 = 1.0으로 정규화)
q_factor = (mach_range / 2.0) ** 2   # q ∝ V^2 ∝ Mach^2 (고도 고정)

# Mach 2.0 기준 공력 미분계수
M_alpha_base = -200.0
M_q_base     = -4.0
M_delta_base = -80.0
Z_alpha_base = -3.0

# 마하수별 미분계수 (동압 스케일링)
M_alpha_arr = M_alpha_base * q_factor
M_delta_arr = M_delta_base * q_factor
Z_alpha_arr = Z_alpha_base * q_factor
V_arr       = mach_range * a_sound_SL

# 3-loop 이득 계산
Kg_arr      = (-2.0 * zeta_3L * omega_3L - M_q_base) / M_delta_arr
K_omega_arr = 0.1 * np.abs(Kg_arr)
KA_arr      = 1.0 / (tau_3L * np.abs(Z_alpha_arr) * V_arr)

print("=== 마하수별 이득 스케줄 ===")
print(f"{'Mach':>6} {'Kg':>10} {'K_omega':>10} {'KA':>12} {'M_α':>10}")
print("-" * 55)
for i, M in enumerate(mach_range):
    print(f"  {M:.1f}  {Kg_arr[i]:10.5f}  {K_omega_arr[i]:10.5f}  "
          f"{KA_arr[i]:12.7f}  {M_alpha_arr[i]:10.1f}")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('마하수에 따른 이득 스케줄 (Gain Schedule vs Mach Number)', fontsize=13)

axes[0, 0].plot(mach_range, Kg_arr, 'b-o', linewidth=2, markersize=6)
axes[0, 0].set_ylabel('Kg (레이트 이득)', fontsize=10)
axes[0, 0].set_title('내부 루프 이득 Kg')

axes[0, 1].plot(mach_range, K_omega_arr, 'r-o', linewidth=2, markersize=6)
axes[0, 1].set_ylabel('K_ω (synthetic stability)', fontsize=10)
axes[0, 1].set_title('중간 루프 이득 K_ω')

axes[1, 0].plot(mach_range, KA_arr * 1e4, 'g-o', linewidth=2, markersize=6)
axes[1, 0].set_ylabel('KA × 10⁴ (가속도 이득)', fontsize=10)
axes[1, 0].set_title('외부 루프 이득 KA')
axes[1, 0].set_xlabel('마하수 (Mach)', fontsize=10)

axes[1, 1].plot(mach_range, np.abs(M_alpha_arr), 'm-o', linewidth=2, markersize=6)
axes[1, 1].set_ylabel('|M_α| (/s²)', fontsize=10)
axes[1, 1].set_title('정적 안정성 미분계수 |M_α|')
axes[1, 1].set_xlabel('마하수 (Mach)', fontsize=10)

for ax in axes.flat:
    ax.set_xlabel('마하수 (Mach)', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0.8, 4.2])

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# 마하수별 계단 응답 비교 (M=1.5, 2.0, 3.0)
# ---------------------------------------------------------------

mach_test = [1.5, 2.0, 3.0]
colors_m  = ['#1f77b4', '#ff7f0e', '#2ca02c']
t_cmp     = np.linspace(0, 0.8, 4000)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('마하수별 3-Loop Autopilot 응답 (Gain Scheduling 효과)', fontsize=13)

for mach_i, col in zip(mach_test, colors_m):
    # 해당 마하수의 공력 미분계수
    qf_i      = (mach_i / 2.0) ** 2
    M_al_i    = M_alpha_base * qf_i
    M_del_i   = M_delta_base * qf_i
    Z_al_i    = Z_alpha_base * qf_i
    V_i       = mach_i * a_sound_SL

    # 이득 계산 (스케줄된 이득)
    Kg_i      = (-2.0 * zeta_3L * omega_3L - M_q_base) / M_del_i
    K_om_i    = 0.1 * abs(Kg_i)
    KA_i      = 1.0 / (tau_3L * abs(Z_al_i) * V_i)

    # 내부+중간 루프 닫힌 후 분모
    K_tot_i   = Kg_i + K_om_i
    M_q_eff_i = M_q_base + M_del_i * K_tot_i
    a1_i      = -(Z_al_i + M_q_eff_i)
    a0_i      = Z_al_i * M_q_eff_i - M_al_i

    # 기체 TF (분자 계수 업데이트)
    b1_i = V_i * (Z_delta)
    b0_i = V_i * (-Z_delta * M_q_base + M_del_i * Z_al_i - M_al_i * Z_delta)
    TF_inner_i = TransferFunction([b1_i, b0_i], [1.0, a1_i, a0_i])

    # 외부 루프 PI
    TF_out_i = TransferFunction([KA_i * KDC, Ki_3L], [1.0, 0.0])

    # 개루프 → 닫힌 루프
    OL_i  = signal.series(TF_out_i.num, TF_out_i.den, TF_inner_i.num, TF_inner_i.den)
    CL_i  = signal.feedback(OL_i[0], OL_i[1], sign=-1)

    try:
        t_i, y_i = signal.step(TransferFunction(CL_i[0], CL_i[1]), T=t_cmp)
        y_i_scaled = y_i * a_cmd_val

        axes[0].plot(t_i * 1000, y_i_scaled, color=col, linewidth=2, label=f'Mach {mach_i}')

        # 보드 선도 (대역폭)
        w_i, mag_i, _ = bode(TransferFunction(CL_i[0], CL_i[1]), w=w_cmp)
        axes[1].semilogx(w_i, mag_i, color=col, linewidth=2, label=f'Mach {mach_i}')
    except Exception as e:
        print(f"Mach {mach_i} 계산 오류: {e}")

axes[0].axhline(a_cmd_val, color='k', linestyle='--', linewidth=1.5, label='명령')
axes[0].set_xlabel('시간 (ms)', fontsize=11)
axes[0].set_ylabel('법선 가속도 (m/s²)', fontsize=11)
axes[0].set_title('계단 응답 (Gain Scheduling 적용)', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].set_xlim([0, 600])

axes[1].axhline(-3, color='gray', linestyle='--', alpha=0.7, label='-3 dB')
axes[1].set_xlabel('주파수 (rad/s)', fontsize=11)
axes[1].set_ylabel('크기 (dB)', fontsize=11)
axes[1].set_title('닫힌 루프 보드 선도 (마하수별)', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].set_xlim([0.1, 500])
axes[1].set_ylim([-40, 10])

for ax in axes:
    ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. 시간영역 시뮬레이션 (Time-Domain Simulation)

지금까지의 선형 해석을 검증하기 위해, 다음 구성요소를 직렬 연결한 **비선형 시간영역 시뮬레이션**을 수행합니다.

```
a_cmd → [3-Loop Autopilot] → δ_cmd → [구동기 (위치/속도 포화)] → δ → [기체 (RK4)] → a_z
                    ↑ q ──────────────────────────────────────────────────────────────┘
                    ↑ a_z ────────────────────────────────────────────────────────────┘
```

비선형 효과:
- 구동기 위치 포화: $|\delta| \leq 25°$
- 구동기 속도 포화: $|\dot{\delta}| \leq 300°/s$
- 적분기 Anti-windup

In [ ]:
# ---------------------------------------------------------------
# 완전 비선형 시간영역 시뮬레이션
# autopilot.py + actuator.py 로직 직접 구현
# ---------------------------------------------------------------

class Autopilot3Loop:
    """3-Loop 오토파일럿 (이산 시간 구현)"""
    def __init__(self, Kg, K_omega, KA, KDC=1.0, Ki=0.02, int_limit=0.3):
        self.Kg, self.K_omega = Kg, K_omega
        self.KA, self.KDC    = KA, KDC
        self.Ki              = Ki
        self.int_limit       = int_limit
        self.integral        = 0.0

    def compute(self, a_cmd, a_meas, q_meas, dt):
        a_err = self.KDC * a_cmd - a_meas
        self.integral += a_err * dt
        clamp = self.int_limit / self.Ki if self.Ki != 0 else 1e9
        self.integral = np.clip(self.integral, -clamp, clamp)
        delta_outer = self.KA * a_err + self.Ki * self.integral
        delta_cmd   = delta_outer + (self.K_omega + self.Kg) * q_meas
        return delta_cmd

class Actuator2nd:
    """2차 구동기 (위치/속도 포화 포함)"""
    def __init__(self, wn, zeta, d_max, r_max):
        self.wn, self.zeta = wn, zeta
        self.d_max, self.r_max = d_max, r_max
        self.d = 0.0; self.dd = 0.0

    def update(self, cmd, dt):
        def deriv(d_, dd_):
            return dd_, self.wn**2 * (cmd - d_) - 2*self.zeta*self.wn*dd_
        k1d, k1dd = deriv(self.d, self.dd)
        k2d, k2dd = deriv(self.d + 0.5*dt*k1d, np.clip(self.dd + 0.5*dt*k1dd, -self.r_max, self.r_max))
        k3d, k3dd = deriv(self.d + 0.5*dt*k2d, np.clip(self.dd + 0.5*dt*k2dd, -self.r_max, self.r_max))
        k4d, k4dd = deriv(self.d + dt*k3d,     np.clip(self.dd + dt*k3dd,     -self.r_max, self.r_max))
        self.d  = np.clip(self.d  + (dt/6)*(k1d  + 2*k2d  + 2*k3d  + k4d),  -self.d_max, self.d_max)
        self.dd = np.clip(self.dd + (dt/6)*(k1dd + 2*k2dd + 2*k3dd + k4dd), -self.r_max, self.r_max)
        return self.d

class Airframe2DOF:
    """단주기 기체 (RK4 적분)"""
    def __init__(self, M_al, M_q, M_del, Z_al, Z_del, V):
        self.M_al, self.M_q, self.M_del = M_al, M_q, M_del
        self.Z_al, self.Z_del, self.V   = Z_al, Z_del, V
        self.alpha = 0.0; self.q = 0.0

    def deriv(self, al, q_, d):
        d_al = self.Z_al * al + q_ + self.Z_del * d
        d_q  = self.M_al * al + self.M_q * q_ + self.M_del * d
        return d_al, d_q

    def update(self, delta, dt):
        k1a, k1q = self.deriv(self.alpha, self.q, delta)
        k2a, k2q = self.deriv(self.alpha + 0.5*dt*k1a, self.q + 0.5*dt*k1q, delta)
        k3a, k3q = self.deriv(self.alpha + 0.5*dt*k2a, self.q + 0.5*dt*k2q, delta)
        k4a, k4q = self.deriv(self.alpha + dt*k3a,     self.q + dt*k3q,     delta)
        self.alpha += (dt/6) * (k1a + 2*k2a + 2*k3a + k4a)
        self.q     += (dt/6) * (k1q + 2*k2q + 2*k3q + k4q)
        az = self.V * (self.Z_al * self.alpha + self.Z_del * delta)
        return self.q, az

# ---------------------------------------------------------------
# 시뮬레이션 실행
# ---------------------------------------------------------------
dt   = 0.001   # 1 ms 시간 간격
t_sim = np.arange(0, 1.0, dt)
N     = len(t_sim)

# 가속도 명령 시나리오: 소형 + 대형 계단
a_cmd_sim = np.where(t_sim < 0.5, 40.0, 80.0)  # 0~0.5s: 40 m/s², 0.5s~: 80 m/s² (>포화)

# 컴포넌트 초기화 (Mach 2.0 이득)
ap  = Autopilot3Loop(Kg=Kg, K_omega=K_omega, KA=KA, Ki=Ki_3L)
act = Actuator2nd(wn=wn_act, zeta=zeta_act, d_max=delta_max, r_max=rate_max)
af  = Airframe2DOF(M_alpha, M_q, M_delta, Z_alpha, Z_delta, V)

# 기록 배열
rec_az  = np.zeros(N)
rec_q   = np.zeros(N)
rec_d   = np.zeros(N)
rec_cmd = np.zeros(N)

az_meas = 0.0
q_meas  = 0.0

for k in range(N):
    # 오토파일럿: 핀 명령 생성
    delta_cmd = ap.compute(a_cmd_sim[k], az_meas, q_meas, dt)
    # 구동기: 실제 핀 편향
    delta_act = act.update(delta_cmd, dt)
    # 기체: 피치율 및 법선 가속도
    q_meas, az_meas = af.update(delta_act, dt)
    # 기록
    rec_cmd[k] = a_cmd_sim[k]
    rec_az[k]  = az_meas
    rec_q[k]   = np.degrees(q_meas)
    rec_d[k]   = np.degrees(delta_act)

print(f"시뮬레이션 완료: {N} 스텝 × {dt*1000:.1f}ms")
print(f"최대 법선 가속도: {np.max(np.abs(rec_az)):.1f} m/s²")
print(f"최대 핀 편향: {np.max(np.abs(rec_d)):.1f}°  (한계: {np.degrees(delta_max):.0f}°)")

# 플롯
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle('3-Loop Autopilot 비선형 시간영역 시뮬레이션', fontsize=13)

axes[0].plot(t_sim * 1000, rec_cmd, 'r--', linewidth=1.5, label='가속도 명령')
axes[0].plot(t_sim * 1000, rec_az,  'b-',  linewidth=1.5, label='달성 가속도')
axes[0].set_ylabel('법선 가속도 (m/s²)', fontsize=10)
axes[0].set_title('가속도 명령 추종', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].axvline(500, color='gray', linestyle=':', alpha=0.6, label='명령 변환')

axes[1].plot(t_sim * 1000, rec_d, 'g-', linewidth=1.5, label='핀 편향각')
axes[1].axhline(np.degrees(delta_max),  color='r', linestyle='--', alpha=0.7, label=f'+한계 ({np.degrees(delta_max):.0f}°)')
axes[1].axhline(-np.degrees(delta_max), color='r', linestyle='--', alpha=0.7, label=f'-한계')
axes[1].set_ylabel('핀 편향각 (deg)', fontsize=10)
axes[1].set_title('구동기 응답 (포화 포함)', fontsize=11)
axes[1].legend(fontsize=9)

axes[2].plot(t_sim * 1000, rec_q, 'm-', linewidth=1.5, label='피치율 q')
axes[2].set_xlabel('시간 (ms)', fontsize=11)
axes[2].set_ylabel('피치율 (deg/s)', fontsize=10)
axes[2].set_title('피치율 응답', fontsize=11)
axes[2].legend(fontsize=10)
axes[2].axvline(500, color='gray', linestyle=':', alpha=0.6)

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. 정리 및 느낀점

### 2-Loop vs 3-Loop Autopilot 비교

| 항목 | 2-Loop | 3-Loop |
|------|--------|--------|
| 루프 수 | 2 (rate + accel) | 3 (rate + synthetic + accel) |
| 적용 대상 | 정적 안정 미사일 | 정적 불안정 포함 |
| 설계 복잡도 | 낮음 | 중간 |
| 이득 수 | $K_q, K_p, K_i$ (3개) | $K_g, K_\omega, K_A, K_i$ (4개) |
| 감쇠 증대 | 내부 루프 1개 | 내부 + 중간 (2중 감쇠) |
| Synthetic stability | 없음 | $K_\omega$가 제공 |
| 이득 설계 | 시행착오 / 수동 | 공식 기반 (Blakelock) |
| 참고 문헌 | Blakelock Ch.7 | Garnell Ch.6, Abd-Elatif et al. |

### 핵심 설계 원칙

1. **대역폭 분리 규칙**: $\omega_{act} \geq 3\omega_{ap} \geq 3\omega_{guidance}$  
   → 각 루프가 상위 루프의 명령을 충실히 추종할 수 있어야 함

2. **구동기는 성능의 병목**: 아무리 우수한 제어 알고리즘도 구동기 대역폭을 초과할 수 없음

3. **Gain Scheduling의 실전적 중요성**:  
   - 고정 이득(fixed gain)으로는 전 비행 포락선(flight envelope)에서 일관된 성능 불가  
   - 마하수·고도 변화 → 동압 변화 → 공력 미분계수 변화 → 이득도 함께 변해야 함  
   - 실제 시스템에서는 실시간 쌍선형 보간(bilinear interpolation)으로 구현

4. **실제 설계의 추가 고려사항**:  
   - **Robust stability**: 공력 불확도, 파라미터 변동에 대한 안정성 여유 확보  
   - **구조 하중 필터 (Structural notch filter)**: 동체 구조 공진 주파수 차단  
   - **Anti-windup**: 구동기 포화 시 적분기 발산 방지  
   - **이산 시간 구현 (Discrete-time)**: 연속 설계 → 이산화 시 위상 지연 증가 고려

### 학습 소감

이번 실습을 통해 오토파일럿 설계가 단순히 제어기를 달아놓는 것이 아니라,  
**유도탄 기체 동역학을 깊이 이해하고 그 특성에 맞춰 각 루프를 체계적으로 설계해야 한다**는 점을 배웠습니다.

특히 3-loop의 synthetic stability 개념은 정적 불안정 유도탄을 능동 안정화하는 핵심 아이디어로,  
Garnell 교수의 통찰이 실제 유도탄 설계에 그대로 적용된다는 점이 인상적이었습니다.

Gain scheduling은 이론보다 실전이 훨씬 복잡하지만,  
결국 **정확한 공력 모델 + 체계적인 이득 설계 공식 + 신뢰성 높은 보간**이 핵심임을 확인했습니다.

---
*참고: 본 노트북은 LIG넥스원 유도조종기법 팀 지원을 위한 학습 포트폴리오입니다.*  
*실제 시스템 설계에는 분류된 비행 데이터 및 CFT/HILS 검증이 추가로 필요합니다.*